# B2 · 自动求导

**学完这一节你应该能**
- 说出 `backward()` 到底做了什么
- 知道梯度存在哪里、怎么读出来
- 知道为什么每步训练都要把梯度清零
- 知道 `no_grad()` 和 `detach()` 分别用在哪

In [ ]:
import torch


def check(name, ok, hint=""):
    if ok:
        print(f"✓ {name}")
    else:
        print(f"✗ {name}")
        if hint:
            print(f"   提示：{hint}")

## 1. 梯度是什么

一句话：**梯度告诉你「参数往哪边调、调多少」，loss 才会变小。**

导数 / 梯度是同一个东西，只是变量多了叫梯度。先看最简单的：`y = x²`，它的导数是 `2x`。在 `x = 3` 处，导数等于 6。

In [ ]:
x = torch.tensor(3.0, requires_grad=True)   # requires_grad=True：让 PyTorch 开始记账
y = x ** 2

y.backward()        # 反向传播，算出 dy/dx

print("y       =", y.item())
print("x.grad  =", x.grad.item(), "（手算 2x = 6）")

`requires_grad=True` 是个开关。关掉它，PyTorch 就不会为这个张量记录计算过程，`backward()` 会直接报错。

In [ ]:
t = torch.tensor(3.0)          # 没有 requires_grad
try:
    (t ** 2).backward()
except Exception as e:
    print(type(e).__name__, ":", str(e).splitlines()[0])

## 2. 计算图

每次对 `requires_grad=True` 的张量做运算，PyTorch 就在背后记一条边，最后连成一张图。`backward()` 就是沿着这张图从后往前走一遍，用链式法则把导数算出来。

每个结果张量的 `grad_fn` 就是「我是怎么算出来的」这条线索。

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x * 3
z = y ** 2

print("z        =", z.item())
print("z.grad_fn        :", type(z.grad_fn).__name__, " ← 我是幂运算来的")
print("它的上游          :", type(z.grad_fn.next_functions[0][0]).__name__, " ← 上游是乘法")
print("x 是叶子节点      :", x.is_leaf)
print("y 不是叶子节点    :", y.is_leaf, " ← 中间结果，backward 之后 y.grad 是 None")

## 3. 梯度存在哪里

存在**每个参数的 `.grad` 属性**里。这是后面 `optimizer.step()` 唯一要看的东西。

In [ ]:
w = torch.tensor([1.0, 2.0], requires_grad=True)
loss = (w ** 2).sum()          # loss = w1² + w2²
loss.backward()

print("w      =", w)
print("w.grad =", w.grad, " ⟨⟨ 每个元素的 2w")

### 关键：梯度是**累加**的

再 `backward()` 一次，梯度不会覆盖，而是加到原来的上面。

In [ ]:
# 真实的训练里，每一步都会重新算一次 loss（新的前向、新的图）
loss = (w ** 2).sum()
loss.backward()
print("第二次 backward 后 w.grad =", w.grad, " ⟨⟨ 翻倍了，不是重算")

这就是 `optimizer.zero_grad()` 存在的全部原因：**不清零，你就是在拿「历史上所有 batch 的梯度之和」去更新参数**，很快就乱套。

清零有两种写法，等价：

In [ ]:
w = torch.tensor([1.0, 2.0], requires_grad=True)
(w ** 2).sum().backward()
print("第一次   :", w.grad.clone())

w.grad = None                        # 写法一
(w ** 2).sum().backward()
print("清零之后 :", w.grad)

(w ** 2).sum().backward()
w.grad.zero_()                       # 写法二
(w ** 2).sum().backward()
print("再清零   :", w.grad)

## 4. `no_grad()` 和 `detach()`

推理时不需要梯度，硬记计算图纯属浪费显存和时间。

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
y = x * 2
print("普通运算      : requires_grad =", y.requires_grad)

with torch.no_grad():
    z = x * 2
print("no_grad 里面  : requires_grad =", z.requires_grad)

d = y.detach()
print("detach 之后   : requires_grad =", d.requires_grad, " ← 和原张量共享数据，但脱离了图")

print("\nno_grad 是「临时关掉」，detach 是「从图上摘下来一个张量」。")

## 5. 对拍：自动求导可信吗

三个办法验证：手算、和官方实现比、**数值差分**。

数值差分不依赖任何框架：给参数加一点点、减一点点，看函数值变多少。

In [ ]:
# 办法一：手算
x = torch.tensor(2.0, requires_grad=True)
(x ** 3).backward()
print("y = x³ 在 x=2 ：autograd =", x.grad.item(), "，手算 3x² =", 3 * 2.0 ** 2)

# 办法二：换个函数
x = torch.tensor(0.5, requires_grad=True)
torch.sin(x).backward()
print("y = sin(x) 在 0.5：autograd =", round(x.grad.item(), 6), "，手算 cos(0.5) =", round(torch.cos(torch.tensor(0.5)).item(), 6))

In [ ]:
# 办法三：数值差分（中心差分）
def numeric_grad(f, x, eps=1e-4):
    return (f(x + eps) - f(x - eps)) / (2 * eps)


f = lambda t: t ** 3
x = torch.tensor(1.5, requires_grad=True)
f(x).backward()

print("autograd     :", round(x.grad.item(), 6))
print("数值差分     :", round(numeric_grad(f, torch.tensor(1.5)).item(), 6))
print("\n这个技巧在调试自定义算子时非常有用：拿不准梯度对不对，就差一下。")

## 6. 手写一步梯度下降

`optimizer.step()` 内部就是这个，没有别的魔法。

In [ ]:
# 目标：让 w 尽量接近 3，loss = (w - 3)²
w = torch.tensor(0.0, requires_grad=True)
lr = 0.1

for i in range(20):
    loss = (w - 3) ** 2
    loss.backward()                  # 算梯度：dloss/dw = 2(w-3)

    with torch.no_grad():            # 更新参数时不需要建图
        w -= lr * w.grad

    w.grad = None                    # 相当于 zero_grad()

    if i % 5 == 0 or i == 19:
        print(f"step {i:>2} | w = {w.item():.4f} | loss = {loss.item():.6f}")

## 7. 练习

In [ ]:
# 练习 1：y = x³，求 x = 2 处的梯度
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3
# ← 在这里写一行触发反向传播

check("练习 1：x.grad 应为 12", x.grad is not None and abs(x.grad.item() - 12.0) < 1e-5,
      hint="y.backward()")

In [ ]:
# 练习 2：下面这段不清零，预测第二次 backward 后 w.grad 是多少
w = torch.tensor([2.0], requires_grad=True)
(w ** 2).sum().backward()
first = w.grad.item()          # 第一次：2w = 4
(w ** 2).sum().backward()
second = w.grad.item()

answer = first                 # ← 改这一行：预测 second 应该等于多少

check("练习 2：预测累加后的梯度", abs(second - answer) < 1e-5,
      hint="梯度是累加的，第二次会在 4 的基础上再加 4")

In [ ]:
# 练习 3：在 no_grad 下算一个新张量 g，让它的 requires_grad 为 False
x = torch.tensor(2.0, requires_grad=True)

g = x * 2                      # ← 改这一行（用 no_grad 或 detach）

check("练习 3：g.requires_grad 应为 False", g.requires_grad is False,
      hint="with torch.no_grad(): 或 (x * 2).detach()")

In [ ]:
# 练习 4：用数值差分算出 f(x) = x³ 在 x = 1.5 处的梯度
def f(t):
    return t ** 3


# 这里必须用 float64：float32 只有约 7 位有效数字，这个差分算不准（下面有说明）
eps = torch.tensor(1e-4, dtype=torch.float64)
x0 = torch.tensor(1.5, dtype=torch.float64)

num = None                     # ← 改这一行：中心差分

check("练习 4：与真实值 6.75 相差小于 1e-3",
      num is not None and abs(num.item() - 6.75) < 1e-3,
      hint="(f(x0 + eps) - f(x0 - eps)) / (2 * eps)")

### 为什么这里要用 float64

数值差分是「两个很接近的数相减」，会把有效数字吃掉。float32 只有约 7 位有效数字，`eps = 1e-4` 时误差已经大到 2e-3。

实测：

| dtype | eps | 算出来的值 | 误差 |
|---|---|---|---|
| float32 | 1e-3 | 6.750345 | 3e-4 |
| float32 | 1e-4 | 6.752014 | 2e-3 |
| float32 | 1e-6 | 6.437302 | 3e-1 |
| float64 | 1e-4 | 6.75000001 | 1e-8 |

所以 **eps 不是越小越好**：

```
总误差 = 截断误差（∝ eps²） + 舍入误差（∝ 1/eps）
```

太大会有截断误差，太小会被浮点舍入淹没，中间有个最优点。float32 下大约 `1e-3`，float64 下可以到 `1e-6`。


In [ ]:
# 练习 5：手写一步梯度下降
# 当前 w = 1.0，目标是最小化 (w - 3)²，学习率 lr = 0.1
# 问：走一步之后 w 应该是多少？
w = torch.tensor(1.0, requires_grad=True)
lr = 0.1
loss = (w - 3) ** 2
loss.backward()

# ← 在这里更新 w（别忘了用 no_grad）

check("练习 5：w 应更新到约 1.4", abs(w.item() - 1.4) < 1e-4,
      hint="w -= lr * w.grad，其中梯度是 2(w-3) = -4")

## 收尾

回答这三个：

1. `backward()` 之后，梯度存在哪里？中间变量（非叶子）的 `.grad` 为什么是 `None`？
2. 为什么每次 `backward()` 前要清零？不清零会发生什么？
3. `no_grad()` 和 `detach()` 有什么区别，各自用在哪？

下一节：**B3 · nn.Module**。